# Diamond Scraper — Luvansh + IGI Report Verification

This notebook:
1. Scrapes 10 round diamonds (1.8–2.5 ct) from [luvansh.com/shop-diamond](https://www.luvansh.com/shop-diamond)
2. Visits each diamond's detail page to find the IGI report number
3. Goes to the IGI Verify-Your-Report page and extracts proportions from the PDF Report tab
4. Filters diamonds matching ideal proportions:

| Parameter | Target Range |
|---|---|
| L/W Ratio | 1.00–1.02 |
| Table % | 54–58% |
| Depth % | 61.0–62.3% |
| Crown angle | 34.0–35.0° |
| Pavilion angle | 40.6–40.9° |
| Crown height % | 14.0–16.0% |
| Pavilion depth % | 42.5–43.2% |

## 1. Install Dependencies & Setup Chrome

In [ ]:
# ── Install Chrome + Selenium for Colab ───────────────────────────
# google-colab-selenium handles Chrome/ChromeDriver version matching
# automatically, which avoids the Chrome 136+ SessionNotCreatedException.
# The [undetected] extra adds UndetectedChrome for Cloudflare bypass.

!pip install -q "google-colab-selenium[undetected]" pdfplumber requests pandas

import sys, os
os.environ["DISPLAY"] = ":99"
print("Dependencies installed.")

## 2. Imports & Configuration

In [ ]:
import re
import time
import json
import io
import requests
import pdfplumber
import pandas as pd

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException
)

# ── Ideal-cut filter ranges ────────────────────────────────────────
IDEAL_RANGES = {
    "lw_ratio":       (1.00, 1.02),
    "table_pct":      (54.0, 58.0),
    "depth_pct":      (61.0, 62.3),
    "crown_angle":    (34.0, 35.0),
    "pavilion_angle": (40.6, 40.9),
    "crown_height":   (14.0, 16.0),
    "pavilion_depth": (42.5, 43.2),
}

# Number of diamonds to scrape per run
DIAMONDS_TO_SCRAPE = 10

# Carat range filter
CARAT_MIN = 1.8
CARAT_MAX = 2.5

# Shape filter
SHAPE = "Round"

print("Configuration loaded.")
print(f"Shape: {SHAPE}")
print(f"Carat range: {CARAT_MIN}–{CARAT_MAX}")
print(f"Diamonds to scrape: {DIAMONDS_TO_SCRAPE}")
print(f"Ideal ranges: {json.dumps({k: list(v) for k, v in IDEAL_RANGES.items()}, indent=2)}")

## 3. Selenium Helper — Create Headless Chrome Driver

In [ ]:
def create_driver():
    """
    Create a Chrome driver compatible with Google Colab.

    Uses google-colab-selenium which automatically handles Chrome/ChromeDriver
    version matching (fixes the Chrome 136+ SessionNotCreatedException).

    Tries UndetectedChrome first (for Cloudflare bypass), falls back to
    regular Chrome if that fails.
    """
    # ── Attempt 1: google-colab-selenium UndetectedChrome ─────────
    try:
        import google_colab_selenium as gs
        driver = gs.UndetectedChrome()
        driver.implicitly_wait(10)
        print("[Driver] Using google-colab-selenium UndetectedChrome")
        return driver
    except Exception as e1:
        print(f"[Driver] UndetectedChrome failed: {e1}")

    # ── Attempt 2: google-colab-selenium regular Chrome ───────────
    try:
        import google_colab_selenium as gs
        driver = gs.Chrome()
        driver.implicitly_wait(10)
        print("[Driver] Using google-colab-selenium Chrome")
        return driver
    except Exception as e2:
        print(f"[Driver] gs.Chrome failed: {e2}")

    # ── Attempt 3: Manual Selenium + Chromium with explicit binary ─
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.chrome.service import Service

        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
        options.add_argument("--disable-blink-features=AutomationControlled")
        # Try to set the binary location explicitly
        for binary in ["/usr/bin/google-chrome-stable", "/usr/bin/google-chrome",
                       "/usr/bin/chromium-browser", "/usr/bin/chromium"]:
            if os.path.exists(binary):
                options.binary_location = binary
                break

        # Try common chromedriver locations
        for drv_path in ["/usr/bin/chromedriver", "/usr/local/bin/chromedriver",
                         "/usr/lib/chromium-browser/chromedriver"]:
            if os.path.exists(drv_path):
                service = Service(drv_path)
                driver = webdriver.Chrome(service=service, options=options)
                driver.implicitly_wait(10)
                print(f"[Driver] Using manual Selenium + {binary}")
                return driver
    except Exception as e3:
        print(f"[Driver] Manual Selenium failed: {e3}")

    # ── Attempt 4: Install Google Chrome fresh + webdriver-manager ─
    print("[Driver] Attempting fresh Google Chrome install ...")
    import subprocess
    subprocess.run([
        "bash", "-c",
        "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb && "
        "dpkg -i google-chrome-stable_current_amd64.deb 2>/dev/null; "
        "apt-get -f install -y -qq 2>/dev/null; "
        "pip install -q webdriver-manager"
    ], capture_output=True)

    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.binary_location = "/usr/bin/google-chrome-stable"

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.implicitly_wait(10)
    print("[Driver] Using fresh Chrome install + webdriver-manager")
    return driver


# Quick test
driver = create_driver()
print(f"Chrome driver ready — {driver.capabilities.get('browserVersion', 'unknown')}")
driver.quit()
print("Driver test passed!")

## 4. Step 1 — Scrape Diamond Listings from Luvansh

**How Luvansh's page works** (ASP.NET MVC + jQuery AJAX):
- On `$(document).ready()`, the page calls `filterShopDiamondTable(1)` which does an AJAX `GET` to `/Shop/_ShopDiamondTable` with a `filterVM` object
- The response replaces `#dtInventory tbody` with new `<tr class="diamondList">` rows
- Shape filters are **checkboxes** (`#chkRound`, `#chkOval`, etc.) — NOT buttons
- Carat/color use **ionRangeSlider** jQuery plugins — NOT plain text inputs
- Scrolling to 80% page height triggers pagination via `filterShopDiamondTable(pageNo+1)`
- Clicking any filter triggers `filterShopDiamondTable(1)` which empties and replaces the table

**Strategy**: Use JavaScript `document.getElementById('chkRound').click()` to trigger the shape filter natively (which properly fires the event handler in `diamond-filter-desktop.js`), then let the page's own AJAX handle filtering. Post-filter carat range in Python.

In [ ]:
from bs4 import BeautifulSoup

def scrape_luvansh_from_html(html_path, carat_min=1.8, carat_max=2.5,
                              shape="Round", count=10):
    """
    Parse diamonds from a saved Luvansh HTML file instead of live scraping.

    The HTML contains <tr class="diamondList"> rows with structure:
      <tr onclick="displayDetailView(this, productId)">
        <td> (R) Round </td>        ← Shape (R=Round, F=Fancy/other)
        <td> <del>$569</del> $284 </td>  ← Price
        <td> 1.83 </td>              ← Carat
        <td> F </td>                 ← Color
        <td> VVS2 </td>              ← Clarity
        <td> Ideal </td>             ← Cut
      </tr>
    """
    print(f"[Luvansh] Parsing HTML file: {html_path} ...")
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    rows = soup.select("tr.diamondList")
    print(f"[Luvansh] Found {len(rows)} tr.diamondList rows in HTML file")

    diamonds = []
    skipped_carat = 0
    skipped_shape = 0

    for row in rows:
        # Extract product ID from onclick
        onclick = row.get("onclick", "")
        pid_match = re.search(r'displayDetailView\(this,\s*(\d+)\)', onclick)
        product_id = pid_match.group(1) if pid_match else None

        # Get all <td> cells
        cells = row.find_all("td")
        if len(cells) < 6:
            continue

        # Get text content from each cell (strip inner <a>, <del>, etc.)
        cell_texts = [c.get_text(strip=True) for c in cells]

        # Shape (column 0): "(R) Round" or "(F) Oval" etc.
        shape_text = cell_texts[0]
        if shape.lower() not in shape_text.lower():
            skipped_shape += 1
            continue

        # Carat (column 2)
        carat_val = None
        m = re.match(r'^(\d+\.?\d*)$', cell_texts[2])
        if m:
            carat_val = float(m.group(1))

        if carat_val is None or not (carat_min <= carat_val <= carat_max):
            skipped_carat += 1
            continue

        # Price (column 1): "<del>$569</del> $284" → get discounted price
        price = None
        # Get the discounted price (text after <del>)
        price_td = cells[1]
        del_tag = price_td.find("del")
        if del_tag:
            # Remove the <del> to get remaining text = discounted price
            del_tag.decompose()
            disc_text = price_td.get_text(strip=True)
            disc_match = re.search(r'\$\s*([\d,]+)', disc_text)
            if disc_match:
                price = f"${disc_match.group(1)}"
        if not price:
            all_prices = re.findall(r'\$\s*([\d,]+)', cell_texts[1])
            if all_prices:
                price = f"${all_prices[-1]}"

        # Color (column 3), Clarity (column 4), Cut (column 5)
        color = cell_texts[3] if re.match(r'^[D-Z]$', cell_texts[3]) else None
        clarity = cell_texts[4] if re.match(r'^(?:FL|IF|VVS[12]|VS[12]|SI[12]|I[123])$', cell_texts[4], re.I) else None
        cut_grade = cell_texts[5] if cell_texts[5].lower() in ("ideal", "excellent", "very good", "good", "fair", "poor") else None

        name = f"IGI {carat_val}ct {color or '?'} {clarity or '?'} Round"
        diamond = {
            "name": name,
            "carat": carat_val,
            "product_id": product_id,
            "detail_url": f"https://www.luvansh.com/Shop/_ShopDiamondTableDetail?productId={product_id}" if product_id else None,
            "color": color,
            "clarity": clarity,
            "cut_grade": cut_grade,
            "price": price,
        }
        diamonds.append(diamond)

        if len(diamonds) >= count:
            break

    print(f"[Luvansh] Parsed {len(diamonds)} matching diamonds "
          f"(skipped {skipped_shape} wrong shape, {skipped_carat} out-of-carat-range)")
    return diamonds


# ── Run Step 1: Parse from saved HTML ─────────────────────────────
# The HTML file is uploaded to Colab from the repo
import os

# Try multiple paths (Colab downloads to /content, or it may be in the repo)
html_paths = [
    "/content/New Text Document.txt",
    "New Text Document.txt",
    os.path.join(os.getcwd(), "New Text Document.txt"),
]
html_path = None
for p in html_paths:
    if os.path.exists(p):
        html_path = p
        break

if html_path is None:
    print("ERROR: HTML file not found! Please upload 'New Text Document.txt' to Colab.")
    print("You can do this by:")
    print("  1. Click the folder icon on the left sidebar")
    print("  2. Upload the file to /content/")
    print("  Or run: !wget 'https://raw.githubusercontent.com/idoo25/WIUSI/claude/fix-empty-filter-results-nNrD2/New%20Text%20Document.txt'")
    diamonds = []
else:
    diamonds = scrape_luvansh_from_html(
        html_path, CARAT_MIN, CARAT_MAX, SHAPE, DIAMONDS_TO_SCRAPE
    )
    for i, d in enumerate(diamonds, 1):
        print(f"  {i}. {d['name'][:60]}  |  {d.get('carat', '?')} ct  |  "
              f"ID={d.get('product_id', 'N/A')}  |  {d.get('price', '?')}")
    if not diamonds:
        print("\n[!] No diamonds found. Check the file content.")

## 5. Step 2 — Visit Each Diamond Detail Page → Extract IGI Report Number

In [ ]:
def extract_sku_from_product_page(driver, product_id, carat, color, clarity,
                                   shape="round"):
    """
    Visit the Luvansh product page and extract the SKU number.

    SKU format: LV-ROUNDEVVS2-726536327
    IGI report = "LG" + last part of SKU → LG726536327
    IGI PDF URL = https://api.igi.org/viewpdf.php?r=LG726536327
    """
    # Construct the product page URL slug
    carat_str = str(carat).replace(".", "-")
    slug = (f"igi-certified-{carat_str}-{color.lower()}-"
            f"{clarity.lower()}-{shape.lower()}-lab-created-diamond-r")
    url = f"https://www.luvansh.com/product/{product_id}/{slug}"

    print(f"  [Product] Loading {url} ...")
    driver.get(url)
    time.sleep(4)

    page_source = driver.page_source
    page_text = driver.find_element(By.TAG_NAME, "body").text

    # ── Extract SKU ───────────────────────────────────────────────
    sku = None
    sku_patterns = [
        r'SKU\s*:\s*(LV-[A-Z0-9-]+)',
        r'SKU\s*:\s*([A-Z0-9-]+)',
        r'sku["\s:]+\s*(LV-[A-Z0-9-]+)',
        r'(LV-[A-Z]+-\d+)',
    ]
    for pat in sku_patterns:
        m = re.search(pat, page_source + " " + page_text, re.I)
        if m:
            sku = m.group(1)
            break

    # ── Derive IGI report number from SKU ─────────────────────────
    # SKU: LV-ROUNDEVVS2-726536327 → last part = 726536327 → LG726536327
    igi_report = None
    igi_pdf_url = None
    if sku:
        parts = sku.split("-")
        if len(parts) >= 3:
            sku_number = parts[-1]  # "726536327"
            igi_report = f"LG{sku_number}"
            igi_pdf_url = f"https://api.igi.org/viewpdf.php?r={igi_report}"

    if sku:
        print(f"  [Product] SKU: {sku}")
        print(f"  [Product] IGI Report: {igi_report}  →  PDF: {igi_pdf_url}")
    else:
        print(f"  [Product] SKU not found on page")
        # Debug
        sku_area = re.search(r'(?i)(sku.{0,50})', page_text)
        if sku_area:
            print(f"  [DEBUG] SKU context: {sku_area.group(0)!r}")
        lv_area = re.search(r'(LV-.{0,40})', page_source)
        if lv_area:
            print(f"  [DEBUG] LV- found: {lv_area.group(0)!r}")

    return sku, igi_report, igi_pdf_url


# ── Run Step 2: Visit product pages → extract SKU + IGI ───────────
if diamonds:
    print(f"[Step 2] Extracting SKU from product pages for {len(diamonds)} diamonds ...\n")
    driver = create_driver()
    try:
        for i, d in enumerate(diamonds, 1):
            pid = d.get("product_id")
            color = d.get("color", "")
            clarity = d.get("clarity", "")
            carat = d.get("carat", 0)

            if not pid or not color or not clarity:
                print(f"  {i}. Missing data — skipped")
                d["sku"] = None
                d["igi_report_number"] = None
                d["igi_pdf_url"] = None
                continue

            sku, igi_report, igi_pdf_url = extract_sku_from_product_page(
                driver, pid, carat, color, clarity
            )
            d["sku"] = sku
            d["igi_report_number"] = igi_report
            d["igi_pdf_url"] = igi_pdf_url
            print()

            time.sleep(2)
    finally:
        driver.quit()
else:
    print("[Step 2] No diamonds to process — skipping.")

print("\n── Diamonds with SKU + IGI ──")
for i, d in enumerate(diamonds, 1):
    print(f"  {i}. SKU={d.get('sku', 'N/A'):<30}  "
          f"IGI={d.get('igi_report_number', 'N/A'):<14}  "
          f"{d.get('carat','?')}ct {d.get('color','?')} {d.get('clarity','?')}  "
          f"{d.get('price','?')}")

## 6. Step 3 — Go to IGI Verification → Extract Proportions

**Important notes from research:**
- IGI uses **Cloudflare bot protection** — `undetected-chromedriver` helps bypass this
- The verification page may have tabs: **Report Details**, **4C's**, **Journey**, **Videos**, and possibly **PDF Report**
- Some reports require **carat weight input** as an anti-scraping measure
- Only **full IGI reports** (not card-type certificates) contain proportions data
- The proportions diagram shows values like: `13.5% 58% 33.1° 40.9° 43% Pointed 61%`

In [ ]:
def try_igi_internal_api(report_number):
    """
    Attempt to fetch diamond data from IGI's internal API endpoint.
    This may not always be accessible, but is much faster than Selenium when it works.
    Returns a dict with proportions, or None if the API is unreachable.
    """
    api_url = f"http://20.157.116.47:5000/report/{report_number}"
    try:
        resp = requests.get(api_url, timeout=5, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        })
        if resp.status_code == 200:
            data = resp.json()
            print(f"  [IGI API] Got JSON response for {report_number}")
            # Map JSON keys to our standard keys
            props = {
                "report_number": report_number,
                "table_pct": data.get("table") or data.get("Table"),
                "depth_pct": data.get("depth") or data.get("Depth"),
                "crown_angle": data.get("crown_angle") or data.get("CrownAngle"),
                "pavilion_angle": data.get("pavilion_angle") or data.get("PavilionAngle"),
                "crown_height": data.get("crown_height") or data.get("CrownHeight"),
                "pavilion_depth": data.get("pavilion_depth") or data.get("PavilionDepth"),
                "measurements": data.get("measurements") or data.get("Measurements"),
                "polish": data.get("polish") or data.get("Polish"),
                "symmetry": data.get("symmetry") or data.get("Symmetry"),
                "fluorescence": data.get("fluorescence") or data.get("Fluorescence"),
                "cut_grade": data.get("cut") or data.get("Cut") or data.get("CutGrade"),
                "carat_weight": data.get("carat") or data.get("Carat") or data.get("CaratWeight"),
                "color_grade": data.get("color") or data.get("Color"),
                "clarity_grade": data.get("clarity") or data.get("Clarity"),
                "shape": data.get("shape") or data.get("Shape"),
                "girdle": data.get("girdle") or data.get("Girdle"),
                "culet": data.get("culet") or data.get("Culet"),
                "lw_ratio": None,
            }
            # Calculate L/W from measurements if available
            meas = props.get("measurements")
            if meas:
                dims = re.findall(r'(\d+\.\d+)', str(meas))
                if len(dims) >= 2:
                    l, w = float(dims[0]), float(dims[1])
                    if min(l, w) > 0:
                        props["lw_ratio"] = round(max(l, w) / min(l, w), 3)
            return props
    except (requests.ConnectionError, requests.Timeout):
        pass
    except Exception as e:
        print(f"  [IGI API] Error: {e}")
    return None


def extract_igi_proportions(driver, report_number, carat_weight=None):
    """
    Go to the IGI Verify-Your-Report page for the given report number,
    navigate to the tab with proportions data, and extract values.

    Tries the IGI internal API first (fast), then falls back to Selenium scraping.

    Returns a dict with proportions data.
    """
    # ── Fast path: try IGI internal API ───────────────────────────
    api_result = try_igi_internal_api(report_number)
    if api_result and any(v is not None for k, v in api_result.items() if k != "report_number"):
        return api_result

    # ── Slow path: Selenium scraping ──────────────────────────────
    urls_to_try = [
        f"https://www.igi.org/Verify-Your-Report/?r={report_number}",
        f"https://www.igi.org/verify.php?r={report_number}",
    ]

    for url in urls_to_try:
        print(f"  [IGI] Loading {url} ...")
        driver.get(url)
        time.sleep(6)

        # Check if Cloudflare blocked us
        page_text_lower = driver.page_source.lower()
        if "just a moment" in page_text_lower or "checking your browser" in page_text_lower:
            print("  [IGI] Cloudflare challenge detected — waiting 10s ...")
            time.sleep(10)

        if "report" in driver.page_source.lower() and "verify" in driver.page_source.lower():
            break

    # ── Handle carat weight verification form ─────────────────────
    try:
        carat_input = driver.find_elements(By.CSS_SELECTOR,
            "input[name*='carat' i], input[placeholder*='carat' i], "
            "input[id*='carat' i], input[type='number']")
        if carat_input and carat_weight:
            carat_input[0].clear()
            carat_input[0].send_keys(str(carat_weight))
            submit_btns = driver.find_elements(By.CSS_SELECTOR,
                "button[type='submit'], input[type='submit'], "
                "button.btn-primary, button:not([disabled])")
            for btn in submit_btns:
                btn_text = btn.text.strip().lower()
                if any(w in btn_text for w in ["check", "verify", "submit", "search"]):
                    btn.click()
                    time.sleep(5)
                    break
            print(f"  [IGI] Submitted carat weight: {carat_weight}")
    except Exception as e:
        print(f"  [IGI] Carat verification step: {e}")

    # ── Click through tabs to find proportions ────────────────────
    tab_selectors = [
        (By.ID, "pdf-tab"),
        (By.XPATH, "//button[contains(text(),'PDF')]"),
        (By.XPATH, "//a[contains(text(),'PDF')]"),
        (By.XPATH, "//button[contains(text(),'Report Details')]"),
        (By.XPATH, "//a[contains(text(),'Report Details')]"),
        (By.XPATH, "//button[contains(text(),'Report')]"),
        (By.XPATH, "//*[@role='tab']"),
    ]
    for by, sel in tab_selectors:
        try:
            tab = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((by, sel))
            )
            tab_text = tab.text.strip()
            tab.click()
            time.sleep(3)
            print(f"  [IGI] Clicked tab: '{tab_text}'")
            break
        except (TimeoutException, NoSuchElementException):
            continue

    # ── Extract proportions from page text + HTML ─────────────────
    page_text = driver.find_element(By.TAG_NAME, "body").text
    page_source = driver.page_source
    combined = page_text + "\n" + page_source

    props = {
        "report_number": report_number,
        "measurements": None, "table_pct": None, "depth_pct": None,
        "crown_angle": None, "pavilion_angle": None,
        "crown_height": None, "pavilion_depth": None,
        "girdle": None, "culet": None,
        "polish": None, "symmetry": None, "fluorescence": None,
        "lw_ratio": None, "cut_grade": None,
        "carat_weight": None, "color_grade": None,
        "clarity_grade": None, "shape": None,
    }

    def find_float(pattern, text=combined):
        m = re.search(pattern, text, re.I)
        try: return float(m.group(1)) if m else None
        except ValueError: return None

    def find_str(pattern, text=combined):
        m = re.search(pattern, text, re.I)
        return m.group(1).strip() if m else None

    # Measurements: "8.13 - 8.16 X 4.96 MM"
    meas_m = re.search(
        r'Measurements?\s*:?\s*(\d+\.\d+\s*[-–]\s*\d+\.\d+\s*[Xx×]\s*\d+\.\d+)\s*(?:MM|mm)?',
        combined, re.I)
    if meas_m:
        props["measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["lw_ratio"] = round(max(l, w) / min(l, w), 3)

    # Named field extraction
    props["table_pct"] = find_float(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["depth_pct"] = find_float(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["crown_angle"] = find_float(r'Crown\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[°]?')
    props["pavilion_angle"] = find_float(r'Pavilion\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*[°]?')
    props["crown_height"] = find_float(r'Crown\s*(?:Height)?\s*(?:%?)?\s*:?\s*(\d+\.\d+)\s*%?')
    props["pavilion_depth"] = find_float(r'Pavilion\s*(?:Depth)?\s*(?:%?)?\s*:?\s*(\d+\.\d+)\s*%?')

    # IGI proportions diagram: "13.5% 58% 33.1° 40.9° 43% Pointed 61%"
    prop_pattern = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)[°]\s+(\d+\.\d+)[°]\s+(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        combined)
    if prop_pattern:
        print("  [IGI] Found proportions diagram pattern")
        if props["crown_height"] is None:  props["crown_height"]  = float(prop_pattern.group(1))
        if props["table_pct"] is None:     props["table_pct"]     = float(prop_pattern.group(2))
        if props["crown_angle"] is None:   props["crown_angle"]   = float(prop_pattern.group(3))
        if props["pavilion_angle"] is None:props["pavilion_angle"] = float(prop_pattern.group(4))
        if props["pavilion_depth"] is None:props["pavilion_depth"] = float(prop_pattern.group(5))
        if props["depth_pct"] is None:     props["depth_pct"]     = float(prop_pattern.group(6))

    # Two angles next to each other
    if props["crown_angle"] is None or props["pavilion_angle"] is None:
        angles_m = re.search(r'(\d{2}\.\d+)[°]\s+(\d{2}\.\d+)[°]', combined)
        if angles_m:
            if props["crown_angle"] is None:   props["crown_angle"]   = float(angles_m.group(1))
            if props["pavilion_angle"] is None:props["pavilion_angle"] = float(angles_m.group(2))

    # Electronic copy format: "61% 58% Medium... Pointed EXCELLENT EXCELLENT NONE"
    ec_pattern = re.search(
        r'(\d+(?:\.\d+)?)%\s+(\d+(?:\.\d+)?)%\s+(\w[\w\s]*(?:\(Faceted\))?)\s+'
        r'(Pointed|None|Very\s*Small|Small|Medium|Large)\s+'
        r'(EXCELLENT|VERY\s*GOOD|GOOD)\s+(EXCELLENT|VERY\s*GOOD|GOOD)\s+'
        r'(NONE|FAINT|MEDIUM|STRONG)', combined, re.I)
    if ec_pattern:
        print("  [IGI] Found electronic copy pattern")
        if props["depth_pct"] is None:      props["depth_pct"]     = float(ec_pattern.group(1))
        if props["table_pct"] is None:      props["table_pct"]     = float(ec_pattern.group(2))
        if props["girdle"] is None:         props["girdle"]        = ec_pattern.group(3).strip()
        if props["culet"] is None:          props["culet"]         = ec_pattern.group(4).strip()
        if props["polish"] is None:         props["polish"]        = ec_pattern.group(5).strip()
        if props["symmetry"] is None:       props["symmetry"]      = ec_pattern.group(6).strip()
        if props["fluorescence"] is None:   props["fluorescence"]  = ec_pattern.group(7).strip()

    # Grading fields
    props["cut_grade"]    = props["cut_grade"]    or find_str(r'Cut\s*(?:Grade)?\s*:?\s*(IDEAL|EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["polish"]       = props["polish"]       or find_str(r'Polish\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["symmetry"]     = props["symmetry"]     or find_str(r'Symmetry\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["fluorescence"] = props["fluorescence"] or find_str(r'Fluorescence\s*:?\s*(NONE|FAINT|MEDIUM|STRONG|VERY\s*STRONG)')
    props["girdle"]       = props["girdle"]       or find_str(r'Girdle\s*:?\s*([A-Za-z\s]+(?:\(Faceted\))?)')
    props["culet"]        = props["culet"]        or find_str(r'Culet\s*:?\s*(None|Pointed|Very\s*Small|Small|Medium|Large)')

    carat_v = find_float(r'Carat\s*Weight\s*:?\s*(\d+\.\d+)')
    if carat_v: props["carat_weight"] = carat_v
    props["color_grade"]   = find_str(r'Color\s*Grade\s*:?\s*([A-Z])')
    props["clarity_grade"] = find_str(r'Clarity\s*Grade\s*:?\s*(\w+\s*\d*)')
    props["shape"] = find_str(
        r'(?:Shape|Cutting\s*Style)\s*:?\s*'
        r'(ROUND\s*BRILLIANT|ROUND|PRINCESS|CUSHION|OVAL|EMERALD|PEAR|MARQUISE|HEART|RADIANT)')

    # Try embedded PDF
    try:
        pdf_elements = driver.find_elements(By.CSS_SELECTOR,
            "iframe[src*='.pdf'], embed[src*='.pdf'], object[data*='.pdf'], a[href*='.pdf']")
        for el in pdf_elements:
            pdf_url = el.get_attribute("src") or el.get_attribute("data") or el.get_attribute("href")
            if pdf_url:
                print(f"  [IGI] Found PDF URL: {pdf_url}")
                try:
                    pdf_props = extract_from_pdf(pdf_url)
                    for k, v in pdf_props.items():
                        if v is not None and props.get(k) is None:
                            props[k] = v
                except Exception as pdf_e:
                    print(f"  [IGI] PDF extraction failed: {pdf_e}")
                break
    except Exception:
        pass

    print(f"  [IGI] Extracted: Table={props['table_pct']}% Depth={props['depth_pct']}% "
          f"CrAngle={props['crown_angle']}° PavAngle={props['pavilion_angle']}° "
          f"CrHeight={props['crown_height']}% PavDepth={props['pavilion_depth']}% "
          f"L/W={props['lw_ratio']}")
    return props


def extract_from_pdf(pdf_url):
    """Download an IGI PDF report and extract proportions using pdfplumber."""
    props = {}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    resp = requests.get(pdf_url, headers=headers, timeout=30)
    resp.raise_for_status()

    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text() or ""

    def find_float(pattern):
        m = re.search(pattern, full_text, re.I)
        return float(m.group(1)) if m else None

    props["table_pct"]      = find_float(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["depth_pct"]      = find_float(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%?')
    props["crown_angle"]    = find_float(r'Crown\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*°?')
    props["pavilion_angle"] = find_float(r'Pavilion\s*(?:Angle)?\s*:?\s*(\d+\.\d+)\s*°?')
    props["crown_height"]   = find_float(r'Crown\s*(?:Height)?\s*:?\s*(\d+\.\d+)\s*%?')
    props["pavilion_depth"] = find_float(r'Pavilion\s*(?:Depth)?\s*:?\s*(\d+\.\d+)\s*%?')

    # Proportions diagram pattern
    prop_m = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)°\s+(\d+\.\d+)°\s+(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        full_text)
    if prop_m:
        props.setdefault("crown_height",  float(prop_m.group(1)))
        props.setdefault("table_pct",     float(prop_m.group(2)))
        props.setdefault("crown_angle",   float(prop_m.group(3)))
        props.setdefault("pavilion_angle",float(prop_m.group(4)))
        props.setdefault("pavilion_depth",float(prop_m.group(5)))
        props.setdefault("depth_pct",     float(prop_m.group(6)))

    meas_m = re.search(r'(\d+\.\d+\s*[-–]\s*\d+\.\d+\s*[Xx×]\s*\d+\.\d+)', full_text)
    if meas_m:
        props["measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["lw_ratio"] = round(max(l, w) / min(l, w), 3)
    return props

print("IGI extraction functions defined.")

In [ ]:
# ── Run Step 3: Download IGI PDFs and extract proportions ─────────
# Uses the direct PDF API: https://api.igi.org/viewpdf.php?r=LG726536327

for i, d in enumerate(diamonds, 1):
    igi_report = d.get("igi_report_number")
    pdf_url = d.get("igi_pdf_url")

    if not pdf_url:
        print(f"  {i}. No IGI PDF URL — skipped")
        d["igi_props"] = {}
        continue

    print(f"  {i}. Downloading PDF: {pdf_url} ...")
    try:
        props = extract_from_pdf(pdf_url)
        props["report_number"] = igi_report

        # Print what we found
        print(f"     Table={props.get('table_pct','?')}%  "
              f"Depth={props.get('depth_pct','?')}%  "
              f"CrAngle={props.get('crown_angle','?')}°  "
              f"PavAngle={props.get('pavilion_angle','?')}°  "
              f"CrHt={props.get('crown_height','?')}%  "
              f"PavDp={props.get('pavilion_depth','?')}%  "
              f"L/W={props.get('lw_ratio','?')}")

        d["igi_props"] = props

        # Merge carat if found from IGI
        if props.get("carat_weight") and not d.get("carat"):
            d["carat"] = props["carat_weight"]

    except Exception as e:
        print(f"     PDF extraction failed: {e}")
        d["igi_props"] = {"report_number": igi_report}

    time.sleep(1)  # Be polite to IGI API

print("\n── IGI data collection complete ──")
for i, d in enumerate(diamonds, 1):
    p = d.get("igi_props", {})
    print(f"  {i}. {d.get('igi_report_number','?'):<14}  "
          f"Table={p.get('table_pct','?')}%  Depth={p.get('depth_pct','?')}%  "
          f"CrAngle={p.get('crown_angle','?')}°  PavAngle={p.get('pavilion_angle','?')}°  "
          f"L/W={p.get('lw_ratio','?')}")

## 7. Step 4 — Filter Diamonds by Ideal Proportions

In [ ]:
def check_ideal(props, ranges=IDEAL_RANGES):
    """
    Check which ideal-cut criteria a diamond meets.
    Returns (passes_all: bool, details: dict)
    """
    details = {}
    all_pass = True
    any_data = False

    for key, (lo, hi) in ranges.items():
        val = props.get(key)
        if val is None:
            details[key] = {"value": None, "pass": None, "range": f"{lo}–{hi}", "status": "NO DATA"}
            all_pass = False
            continue

        any_data = True
        in_range = lo <= val <= hi
        details[key] = {
            "value": val,
            "pass": in_range,
            "range": f"{lo}–{hi}",
            "status": "✓" if in_range else "✗"
        }
        if not in_range:
            all_pass = False

    return all_pass and any_data, details


# ── Apply filter ──────────────────────────────────────────────────
ideal_diamonds = []
near_ideal_diamonds = []

print("═" * 90)
print(f"{'#':>2}  {'Report':<14} {'Carat':>5}  {'Table%':>6}  {'Depth%':>6}  "
      f"{'CrAngle':>7}  {'PavAngle':>8}  {'CrHt%':>5}  {'PavDp%':>6}  {'L/W':>5}  {'Verdict'}")
print("═" * 90)

for i, d in enumerate(diamonds, 1):
    p = d.get("igi_props", {})
    passes, details = check_ideal(p)

    # Count how many criteria pass
    n_pass = sum(1 for v in details.values() if v["pass"] is True)
    n_total = len(IDEAL_RANGES)

    verdict = f"{n_pass}/{n_total}"
    if passes:
        verdict += " IDEAL"
        ideal_diamonds.append(d)
    elif n_pass >= n_total - 2:
        verdict += " NEAR"
        near_ideal_diamonds.append(d)

    print(f"{i:>2}  {p.get('report_number','?'):<14} {d.get('carat','?'):>5}  "
          f"{str(p.get('table_pct','?')):>6}  {str(p.get('depth_pct','?')):>6}  "
          f"{str(p.get('crown_angle','?')):>7}  {str(p.get('pavilion_angle','?')):>8}  "
          f"{str(p.get('crown_height','?')):>5}  {str(p.get('pavilion_depth','?')):>6}  "
          f"{str(p.get('lw_ratio','?')):>5}  {verdict}")

    # Print per-criterion details
    for key, info in details.items():
        if info["pass"] is False:
            print(f"      ✗ {key}: {info['value']} (need {info['range']})")

print("═" * 90)
print(f"\nResults: {len(ideal_diamonds)} IDEAL, {len(near_ideal_diamonds)} NEAR-IDEAL "
      f"out of {len(diamonds)} diamonds")

## 8. Summary Table (Pandas DataFrame)

In [ ]:
# ── Build a summary DataFrame ─────────────────────────────────────
rows = []
for d in diamonds:
    p = d.get("igi_props", {})
    passes, details = check_ideal(p)
    n_pass = sum(1 for v in details.values() if v["pass"] is True)

    rows.append({
        "IGI Report": p.get("report_number", ""),
        "Carat": d.get("carat"),
        "Color": p.get("color_grade") or d.get("color"),
        "Clarity": p.get("clarity_grade") or d.get("clarity"),
        "Cut": p.get("cut_grade") or d.get("cut_grade"),
        "Shape": p.get("shape"),
        "Table %": p.get("table_pct"),
        "Depth %": p.get("depth_pct"),
        "Crown Angle": p.get("crown_angle"),
        "Pavilion Angle": p.get("pavilion_angle"),
        "Crown Height %": p.get("crown_height"),
        "Pavilion Depth %": p.get("pavilion_depth"),
        "L/W Ratio": p.get("lw_ratio"),
        "Polish": p.get("polish"),
        "Symmetry": p.get("symmetry"),
        "Fluorescence": p.get("fluorescence"),
        "Measurements": p.get("measurements"),
        "Criteria Met": f"{n_pass}/{len(IDEAL_RANGES)}",
        "Ideal?": "YES" if passes else "NO",
        "Luvansh URL": d.get("detail_url", ""),
        "Price": d.get("price", ""),
    })

if not rows:
    print("No diamonds were collected — nothing to display.")
    print("Check the Luvansh scraping step above for errors.")
    df = pd.DataFrame()
else:
    df = pd.DataFrame(rows)

    # Display with highlighting
    def highlight_ideal(row):
        if row.get("Ideal?") == "YES":
            return ["background-color: #c6efce"] * len(row)
        return [""] * len(row)

    display(df.style.apply(highlight_ideal, axis=1).set_caption("Diamond Scraping Results"))

    # Show only ideal diamonds
    if "Ideal?" in df.columns:
        ideal_df = df[df["Ideal?"] == "YES"]
        if not ideal_df.empty:
            print(f"\n{'='*60}")
            print(f"  IDEAL DIAMONDS FOUND: {len(ideal_df)}")
            print(f"{'='*60}")
            display(ideal_df[
                ["IGI Report", "Carat", "Color", "Clarity", "Table %", "Depth %",
                 "Crown Angle", "Pavilion Angle", "Crown Height %", "Pavilion Depth %",
                 "L/W Ratio", "Price", "Luvansh URL"]
            ])
        else:
            print("\nNo diamonds met ALL ideal criteria in this batch.")
            print("Consider widening the ranges or scraping more diamonds.")

## 9. Export Results

In [ ]:
# Save to CSV
csv_filename = "diamond_results.csv"
df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

# In Colab, offer download
try:
    from google.colab import files
    files.download(csv_filename)
except ImportError:
    print(f"(Not in Colab — file saved locally as {csv_filename})")

# Also save the raw data as JSON for debugging
json_filename = "diamond_results.json"
with open(json_filename, "w") as f:
    json.dump(diamonds, f, indent=2, default=str)
print(f"Raw data saved to {json_filename}")

## 10. Ideal Proportions Reference

| Parameter | Target Range | Why |
|---|---|---|
| L/W Ratio | 1.00–1.02 | Looks round; above ~1.03 starts to look slightly oval |
| Table % | 54–58% | Good balance between fire and sparkle |
| Depth % | 61.0–62.3% | Good light return without wasting diameter |
| Crown Angle | 34.0–35.0° | Crown angle that creates fire without hurting light return |
| Pavilion Angle | 40.6–40.9° | Critical for brilliance; outside range risks light leakage |
| Crown Height % | 14.0–16.0% | Usually matches a good crown angle |
| Pavilion Depth % | 42.5–43.2% | Optimal pavilion depth for light performance |